# Bassac River Forecasting — Part 1: Data Prep & Feature Engineering

Combines `config.py`, `1_load_data.py`, `2_prepare_data.py`, `3_features.py`, and leakage checks.

In [1]:
# ==============================================================================
# 1. CONFIGURATION & IMPORTS
# ==============================================================================
from pathlib import Path
import pandas as pd

ROOT_DIR = Path('..').resolve()
DATA_PROCESSED = ROOT_DIR / 'data' / 'processed'

CAL_SLOPE = 0.7936
CAL_INTERCEPT = -0.4725
KOHKHEL_EXTEND_START = '2024-07-01'
TRAIN_END_DATE = '2024-05-26'

FEATURE_COLS = [
    'WL', 'WL_lag1', 'WL_lag2', 'WL_lag3', 'WL_diff',
    'upstream_discharge_lag1', 'upstream_discharge_diff'
]
TARGET_COL = 'WL_next'


In [2]:
# ==============================================================================
# 2. LOAD CLEANED DAILY DATASETS
# ==============================================================================
bassac = pd.read_csv(DATA_PROCESSED / 'bassac_daily.csv', parse_dates=['date']).sort_values('date')
kohkhel = pd.read_csv(DATA_PROCESSED / 'kohkhel_daily.csv', parse_dates=['date']).sort_values('date')
glofas = pd.read_csv(DATA_PROCESSED / 'glofas_daily.csv', parse_dates=['date']).sort_values('date')

print('bassac_daily :', bassac.shape)
print('kohkhel_daily:', kohkhel.shape)
print('glofas_daily :', glofas.shape)


bassac_daily : (1438, 4)
kohkhel_daily: (1498, 2)
glofas_daily : (3235, 2)


In [3]:
# ==============================================================================
# 3. CALIBRATE KOH KHEL, STITCH TIMELINE, AND JOIN GLOFAS
# ==============================================================================
part1 = bassac[['date', 'WL']].copy()
part1['station_source'] = 'Bassac_01'

part2 = kohkhel[kohkhel['date'] >= pd.Timestamp(KOHKHEL_EXTEND_START)].copy()
part2['WL'] = CAL_SLOPE * part2['WL_raw'] + CAL_INTERCEPT
part2['station_source'] = 'Koh_Khel'
part2 = part2[['date', 'WL', 'station_source']]

water_level = pd.concat([part1, part2], ignore_index=True)
end_date = min(water_level['date'].max(), glofas['date'].max())
full_dates = pd.DataFrame({'date': pd.date_range(water_level['date'].min(), end_date, freq='D')})

daily = full_dates.merge(water_level, on='date', how='left')
daily = daily.merge(glofas[['date', 'upstream_discharge']], on='date', how='left')
daily = daily.sort_values('date').reset_index(drop=True)
print('Aligned calendar rows:', len(daily), '| Valid WL:', daily['WL'].notna().sum())


Aligned calendar rows: 2335 | Valid WL: 2176


In [4]:
# ==============================================================================
# 4. BUILD 7 LEAK-FREE FEATURES & TARGET (WL_next)
# ==============================================================================
df = daily.copy()
df['WL_lag1'] = df['WL'].shift(1)
df['WL_lag2'] = df['WL'].shift(2)
df['WL_lag3'] = df['WL'].shift(3)
df['WL_diff'] = df['WL'] - df['WL_lag1']
df['upstream_discharge_lag1'] = df['upstream_discharge'].shift(1)
df['upstream_discharge_diff'] = df['upstream_discharge'] - df['upstream_discharge_lag1']
df[TARGET_COL] = df['WL'].shift(-1)

model_df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
model_df = model_df[['date', 'station_source'] + FEATURE_COLS + [TARGET_COL]]
model_df.to_csv(DATA_PROCESSED / 'modeling_table.csv', index=False)

assert model_df[FEATURE_COLS + [TARGET_COL]].isna().sum().sum() == 0
print('Saved modeling_table.csv:', model_df.shape)
model_df.head()


Saved modeling_table.csv: (2108, 10)


,date,station_source,WL,WL_lag1,WL_lag2,WL_lag3,WL_diff,upstream_discharge_lag1,upstream_discharge_diff,WL_next
0,2020-06-23,Bassac_01,1.273958,1.222813,1.153125,1.123438,0.051146,6917.31,-652.17,1.331354
1,2020-06-24,Bassac_01,1.331354,1.273958,1.222813,1.153125,0.057396,6265.14,-445.11,1.382708
2,2020-06-25,Bassac_01,1.382708,1.331354,1.273958,1.222813,0.051354,5820.03,-261.99,1.389583
3,2020-06-26,Bassac_01,1.389583,1.382708,1.331354,1.273958,0.006875,5558.04,-176.34,1.354688
4,2020-06-27,Bassac_01,1.354688,1.389583,1.382708,1.331354,-0.034896,5381.70,-218.54,1.297292


In [5]:
# ==============================================================================
# 5. LEAKAGE & ALIGNMENT TESTS (4 CHECKS)
# ==============================================================================
assert len(FEATURE_COLS) == 7
for forbidden in ['WL_next', 'WL_0700', 'n_readings', 'date', 'station_source']:
    assert forbidden not in FEATURE_COLS
assert model_df[FEATURE_COLS + [TARGET_COL]].isna().sum().sum() == 0

chk = model_df.sort_values('date').reset_index(drop=True)
consec = chk.index[chk['date'].diff().dt.days == 1]
for idx in consec[:200]:
    assert abs(chk.loc[idx - 1, TARGET_COL] - chk.loc[idx, 'WL']) < 1e-6

cutoff = pd.Timestamp(TRAIN_END_DATE)
assert chk.loc[chk['date'] <= cutoff, 'date'].max() < chk.loc[chk['date'] > cutoff, 'date'].min()
print('All 4 leakage & alignment tests PASSED.')


All 4 leakage & alignment tests PASSED.
